# K-Means, the Elbow and the Silhouette

**DS4DH Practice Pack · Module 06 — Clustering and Segmentation**

*Technique:* K-Means clustering and choosing k

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/06b_kmeans.ipynb)

Data: `merged_dataset.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# This notebook reads the CSVs sitting next to it. In Colab you will be asked
# to upload them from the pack's data/ folder.
NEEDED = ['merged_dataset.csv']

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('merged_dataset.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

K-Means finds groups nobody labelled. You give it a number of clusters; it places
that many centres and assigns each place to the nearest one, iterating until the
assignment stops changing.

The hard part is not the algorithm. It is choosing k, and then deciding whether
the groups mean anything.

In [ ]:
# One row per Census Subdivision.
#   • rows with no csd_code are CMA-level and Canada-level aggregates, not CSDs
#   • each CSD appears 3x (Immigrant / Non-immigrants / Total Immigrant Status)
# Keeping either would silently double- or triple-count places.
csd = df.dropna(subset=['csd_code'])
base = csd[(csd['immigrant_status'] == 'Total Immigrant Status')
           & (csd['cma'].isin(CITIES))].copy()

print(f'{len(df):>4} rows in the raw file')
print(f'{len(csd):>4} after dropping CMA/Canada aggregate rows')
print(f'{len(base):>4} CSDs in the four cities (one row each)')

In [ ]:
FEATURES = ['Total', 'renter_owner_gap', 'tot_income', 'log_pop']
RAW_NEEDED = ['Total', 'renter_owner_gap', 'tot_income', 'tot_pop']

feat = base.dropna(subset=RAW_NEEDED).copy()
feat['log_pop'] = np.log10(feat['tot_pop'])

print(f'{len(base)} CSDs -> {len(feat)} with complete data on all four columns')
print(f'{len(base) - len(feat)} dropped (census suppression in small places)')
print()
for city, n in feat['cma'].value_counts().items():
    print(f'  {city:<11} {n:>3} CSDs')

In [ ]:
RANDOM_STATE = 42
K_RANGE = range(2, 9)

X = StandardScaler().fit_transform(feat[FEATURES])

print(f'{"k":>3}{"inertia":>14}{"silhouette":>14}')
print('-' * 31)
scores = {}
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit(X)
    sil = silhouette_score(X, km.labels_)
    scores[k] = {'inertia': km.inertia_, 'sil': sil}
    print(f'{k:>3}{km.inertia_:>14.1f}{sil:>14.3f}')

## Two ways to choose k

- **Inertia** (within-cluster sum of squares) always falls as k rises — with one
  cluster per point it reaches zero. So you look for the *elbow*: the point after
  which additional clusters stop buying much.
- **Silhouette** measures how much better each point fits its own cluster than the
  next nearest. It has a genuine maximum, so it can be read directly.

Neither is authoritative. They are two pieces of evidence for a judgement call.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ks = list(K_RANGE)

axes[0].plot(ks, [scores[k]['inertia'] for k in ks], 'o-')
axes[0].set_xlabel('k')
axes[0].set_ylabel('inertia (within-cluster SS)')
axes[0].set_title('Elbow — look for the bend')

axes[1].plot(ks, [scores[k]['sil'] for k in ks], 'o-', color='#E8663D')
best = max(ks, key=lambda k: scores[k]['sil'])
axes[1].axvline(best, color='#888', ls=':', label=f'best k = {best}')
axes[1].set_xlabel('k')
axes[1].set_ylabel('silhouette score')
axes[1].set_title('Silhouette — higher is better')
axes[1].legend()
plt.tight_layout()
plt.show()

print(f'silhouette prefers k = {best} (score {scores[best]["sil"]:.3f})')

Silhouette scores in the 0.2–0.4 band mean weak but real structure. This is
normal for social data and worth saying out loud: these are not four crisp
natural kinds of municipality, they are four regions of a continuum that happen
to be a useful way to talk about it.

In [ ]:
N_CLUSTERS = 4

km = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10).fit(X)
feat = feat.assign(cluster=km.labels_)

profile = feat.groupby('cluster')[RAW_NEEDED].mean().round(1)
profile['n'] = feat['cluster'].value_counts().sort_index()
print(profile.to_string())

In [ ]:
# A cluster with no name is not a finding. Name them from the profile.
print('Cluster composition by city:')
print(pd.crosstab(feat['cluster'], feat['cma']).to_string())
print()
for c in sorted(feat['cluster'].unique()):
    g = feat[feat['cluster'] == c]
    print(f'cluster {c}: n={len(g):>3}  STIR {g["Total"].mean():>5.1f}  '
          f'income ${g["tot_income"].mean():>9,.0f}  '
          f'median pop {g["tot_pop"].median():>10,.0f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for c in sorted(feat['cluster'].unique()):
    g = feat[feat['cluster'] == c]
    ax.scatter(g['tot_income'], g['Total'], s=28, alpha=0.75, label=f'cluster {c} (n={len(g)})')
ax.set_xlabel('total household income ($)')
ax.set_ylabel('Total STIR (%)')
ax.set_title(f'K-Means typologies, k={N_CLUSTERS}')
ax.legend()
plt.tight_layout()
plt.show()

### 🔧 Your turn 1

Change `N_CLUSTERS` to 3, then 5, re-running the profile and the scatter.

At which k do the clusters stop being describable in a sentence each? That
limit — not the silhouette score — is usually what should decide k for a typology
meant to be used by people.

### 🔧 Your turn 2

Change `RANDOM_STATE` to 0, then 7, then 123.

Do the cluster *labels* change? Do the cluster *memberships*? Why does
`n_init=10` matter, and what would happen with `n_init=1`?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** At k = 3 the clusters are easy to name — roughly high-burden,
middle, and affluent-low-burden. By k = 5 or 6 at least two clusters differ only
in ways you cannot summarise without reciting the numbers. A typology that cannot
be described is a typology nobody will use, and the silhouette score has no
opinion about that.

**Your turn 2.** The integer labels change freely — cluster 0 under one seed may
be cluster 2 under another. Membership is mostly stable but not perfectly: a
minority of border cases move. `n_init=10` runs the algorithm ten times from
different starts and keeps the best, which is what makes results reproducible
enough to report. With `n_init=1` you get whatever a single random start
converged to, and K-Means converges to local optima routinely.

The deeper point: never present cluster numbers as if they were meaningful
identities. Name them from their profiles and use the names.

</details>

## Where this stops

You have four groups from one algorithm. Whether they reflect structure in the
data or structure in K-Means's assumptions is answered by finding the same groups
a different way — the next notebook.